In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.signal import resample
import json
import warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

## Labels

In [4]:
# root dir
root = 'PD-RS/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,GROUP,ID,EEG,AGE,GENDER,MOCA,UPDRS,TYPE
0,sub-001,PD,1001,PD1001,80,M,19,28.0,1
1,sub-002,PD,1011,PD1011,81,M,17,25.0,1
2,sub-003,PD,1021,PD1021,68,F,26,10.0,1
3,sub-004,PD,1031,PD1031,80,M,22,10.0,1
4,sub-005,PD,1041,PD1041,56,M,21,13.0,1
...,...,...,...,...,...,...,...,...,...
144,sub-145,Control,1451,Control1451,64,F,27,NaN,0
145,sub-146,Control,1461,Control1461,71,M,30,NaN,0
146,sub-147,Control,1471,Control1471,78,M,27,NaN,0
147,sub-148,Control,1481,Control1481,68,F,27,NaN,0


In [6]:
# label mapping: diagnosis code -> label id
label_map = {'PD': 1, 'Control': 0}

# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in participants.itertuples(index=False):
    sub_name = row[0]          # e.g. 'sub-001'
    diag_code = row[1]         # e.g. 'A' / 'F' / 'C'
    pid = int(sub_name[-3:])   # convert '001' -> 1
    label = label_map[diag_code]
    sub_info[sub_name] = (label, pid)

len(sub_info), list(sub_info.items())[:5]

(149,
 [('sub-001', (1, 1)),
  ('sub-002', (1, 2)),
  ('sub-003', (1, 3)),
  ('sub-004', (1, 4)),
  ('sub-005', (1, 5))])

## Features

In [9]:
"""# Test for bad channels, sampling freq and shape
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # print(raw.get_montage())
                # get bad channels
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)"""

Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-001\eeg\sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-002\eeg\sub-002_task-Rest_eeg.fdt
Reading 0 ... 163019  =      0.000 ...   326.038 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-003\eeg\sub-003_task-Rest_eeg.fdt
Reading 0 ... 126179  =      0.000 ...   252.358 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-004\eeg\sub-004_task-Rest_eeg.fdt
Reading 0 ... 131979  =      0.000 ...   263.958 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-005\eeg\sub-005_task-Rest_eeg.fdt
Reading 0 ... 124769  =      0.000 ...   249.538 secs...
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\PD-RS\PD-RS\sub-006\eeg\sub-006_task-Rest_eeg.fdt
Reading 0 ... 131029  =      0.000 ...   262.058 secs...
Reading D:

In [10]:
"""from collections import Counter

print(bad_channel_list)
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
Channel number counter: Counter({63: 119, 64: 29, 66: 1})
Sampling rate counter: Counter({500.0: 149})


In [7]:
def data_preprocessing(
    raw: mne.io.Raw,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=None, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [8]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [9]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
                # For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6; POz is close to Pz
                # So we use T7, T8, P7, P8, POz to replace T3, T4, T5, T6, Pz
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                     'Cz', 'C4', 'T8', 'P7', 'P3', 'POz', 'P4', 'P8', 'O1', 'O2']
                raw.pick(standard_channels)
                raw.reorder_channels(standard_channels)
                source_sample_rate = raw.info['sfreq']
                T_raw = raw.n_times
                for fs in SAMPLE_RATE_LIST:
                    T_res = int(T_raw * fs / source_sample_rate)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub_id][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                    print("----------------------------------------")
        sub_id += 1
    print("-------------------------------------\n")
    
print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

PD-RS/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\PD-RS\PD-RS\sub-001\eeg\sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
fs=200Hz: T_res=56332, STEP=200, n_seg=280
----------------------------------------
fs=100Hz: T_res=28166, STEP=200, n_seg=139
----------------------------------------
fs=50Hz: T_res=14083, STEP=200, n_seg=69
----------------------------------------
-------------------------------------

PD-RS/sub-002\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\PD-RS\PD-RS\sub-002\eeg\sub-002_task-Rest_eeg.fdt
Reading 0 ... 163019  =      0.000 ...   326.038 secs...
fs=200Hz: T_res=65208, STEP=200, n_seg=325
-------------------------

In [10]:
output_root = os.path.join('Processed', sub_folder_path, 'PD-RS')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\PD-RS\X.dat
y path: Processed\L400\PD-RS\y.dat


## PASS 2: Process and store segments into memmap

In [11]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        sub_label, _ = sub_info[sub]
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
                # For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6; POz is close to Pz
                # So we use T7, T8, P7, P8, POz to replace T3, T4, T5, T6, Pz
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                     'Cz', 'C4', 'T8', 'P7', 'P3', 'POz', 'P4', 'P8', 'O1', 'O2']
                raw.pick(standard_channels)
                raw = data_preprocessing(
                    raw=raw,
                    notch_freq=60,  # data collected in United States, so notch at 60 Hz
                    l_freq=0.5,
                    h_freq=45.0,
                    do_bad_interp=True,
                    verbose=True
                )
                raw.reorder_channels(standard_channels)
                source_sample_rate = raw.info['sfreq']
                data = raw.get_data().T  # (T_raw, C)
                T_raw = raw.n_times
                total_seconds_all += data.shape[0] / source_sample_rate
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data_resampled = resample_time_series(data, source_sample_rate, fs)
                    T_res, _ = data_resampled.shape
                    print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(sub_label)       # label
                        y_mm[cur, 1] = float(sub_id)      # Global trial ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
        sub_id += 1
    print("-------------------------------------\n")
    
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

PD-RS/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\PD-RS\PD-RS\sub-001\eeg\sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
✔ Step 2, Montage set: 'standard_1005'.
✔ Step 3: Notch @ 60 Hz
✔ Step 4: Band-pass 0.5–45.0 Hz
ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)
✔ Step 6: Average reference
Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 18 components
Fitting ICA took 3.2s.
✔ ICA fitted.
Applying ICA to Raw instance
    Transforming to ICA space (18 components)
    Zeroing out 6 ICA components
    Projecting back using 19 PCA components
✔ ICA applied. Excluded components: [0, 1, 2, 3, 9, 10]
fs=200Hz

## Load and check the processed data

In [12]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 41174
  T = 400
  C = 19
  X_path = Processed\L400\PD-RS\X.dat
  y_path = Processed\L400\PD-RS\y.dat
-------------------------------------
Subject ID 001: X shape=(488, 400, 19), y shape=(488, 3)
Subject ID 002: X shape=(567, 400, 19), y shape=(567, 3)
Subject ID 003: X shape=(438, 400, 19), y shape=(438, 3)
Subject ID 004: X shape=(456, 400, 19), y shape=(456, 3)
Subject ID 005: X shape=(432, 400, 19), y shape=(432, 3)
Subject ID 006: X shape=(455, 400, 19), y shape=(455, 3)
Subject ID 007: X shape=(414, 400, 19), y shape=(414, 3)
Subject ID 008: X shape=(406, 400, 19), y shape=(406, 3)
Subject ID 009: X shape=(431, 400, 19), y shape=(431, 3)
Subject ID 010: X shape=(595, 400, 19), y shape=(595, 3)
Subject ID 011: X shape=(270, 400, 19), y shape=(270, 3)
Subject ID 012: X shape=(207, 400, 19), y shape=(207, 3)
Subject ID 013: X shape=(211, 400, 19), y shape=(211, 3)
Subject ID 014: X shape=(215, 400, 19), y shape=(215, 3)
Subject ID 015: X shape=(207, 400, 19), y shape=(20